# Linear Learner Binary Classification Demo (Hyperparameter Tuning)

This notebook demonstrates how to train and deploy an AWS SageMaker **Linear Learner** model for a **binary classification** task using **Automatic Model Tuning (Hyperparameter Tuning)**.

### What You Need
- The raw Ames Housing dataset (`AmesHousing.csv`) in this notebook's directory
- An active SageMaker session with appropriate IAM permissions

### What This Notebook Does
1. Loads the raw Ames Housing dataset
2. Preprocesses the data (missing values, outliers, encoding)
3. Creates a binary target variable (above/below median)
4. Splits data into training and validation sets
5. Uploads data to S3
6. Runs a Hyperparameter Tuning job to find the best Linear Learner binary classifier
7. Deploys the best model to a real-time endpoint
8. Evaluates model performance (Accuracy, Precision, Recall, F1)
9. Lets you interactively query the model
10. Cleans up AWS resources

# STEP 1: IMPORT LIBRARIES AND SETUP

In [ ]:
# Install any packages not included in the SageMaker environment
!pip install seaborn -q

In [ ]:
# Standard Libraries
import os
import time
import logging

# Data Handling & Visualization Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Scikit-learn Libraries for Model Building & Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    mean_squared_error, mean_absolute_error, r2_score
)

# AWS & SageMaker Libraries for Model Training and Deployment
import boto3
import sagemaker
from sagemaker import Session, get_execution_role
from sagemaker.estimator import Estimator
from sagemaker.amazon.linear_learner import LinearLearner  # SageMaker's built-in Linear Learner algorithm
from sagemaker.inputs import TrainingInput
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter

# Additional Libraries
from botocore.exceptions import ClientError
from typing import Any, List, Union

# Logger Setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================================
# CONFIGURATION - Update these values for your dataset
# ============================================================
file_path = "AmesHousing.csv"                    # Path to the raw dataset file
target_name = "SalePrice"                        # Name of the original target variable in the dataset
target_col = target_name                          # Initial target column (redefined after binary target creation)
endpoint_name = "binary-learner-endpoint-tuned"  # Name for the SageMaker endpoint

# STEP 2: LOAD THE DATA

In [ ]:
def load_data(filepath):
    """
    Load data from a CSV file into a pandas DataFrame.

    This function reads the CSV file at the given filepath using pandas' read_csv
    method. It includes error handling for common issues such as file not found.

    Parameters:
        filepath (str): The path to the CSV file.

    Returns:
        pd.DataFrame: The DataFrame containing the loaded data.

    Raises:
        FileNotFoundError: If the file does not exist at the specified path.
        Exception: For any other error that occurs during the file reading.
    """
    try:
        # Note: keep_default_na=False prevents pandas from reading the string "None"
        # as NaN. In this dataset, "None" is a valid category (e.g., Mas Vnr Type = None
        # means no masonry veneer). "NA" is still treated as missing (e.g., no basement).
        data = pd.read_csv(filepath, keep_default_na=False, na_values=['', 'NA'])
        return data
    except FileNotFoundError as e:
        print(f"Error: The file at '{filepath}' was not found.")
        raise e
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
        raise e


# Load data using the load_data function.
df = load_data(file_path)
print("Data loaded successfully!\n")

# Display the first few rows of the dataset to provide a quick visual overview.
print("First five rows of the dataset:")
print(df.head())

# Display the dimensions of the dataset (number of rows and columns).
print("\nShape of the dataset:", df.shape)

# Print detailed information about the dataset, including data types and non-null counts.
print("\nDataset Info:")
df.info()

# STEP 3: PREPROCESS THE DATA

## Drop Columns with Too Many Missing Values

In [3]:
def drop_high_missing_columns(dataframe, threshold=0.3):
    """
    Drop columns from the DataFrame where the fraction of missing values exceeds the specified threshold.

    This function calculates the percentage of missing values in each column and removes
    those columns where the fraction of missing values is greater than the threshold. 
    This helps reduce noise and potential bias when features with too many missing values
    might negatively impact the model's performance.

    Parameters:
        dataframe (pd.DataFrame): The input DataFrame to process.
        threshold (float): The maximum allowable fraction of missing values for a column.
                           Columns exceeding this threshold will be dropped (default is 0.3).

    Returns:
        pd.DataFrame: A DataFrame with columns removed that have more than the allowed missing values.
    """
    # Identify columns where the proportion of missing values exceeds the threshold.
    cols_to_drop = dataframe.columns[dataframe.isnull().mean() > threshold]
    
    # Print out the columns that will be dropped for transparency.
    print(f"Dropping columns with more than {threshold*100:.0f}% missing values: {list(cols_to_drop)}")
    
    # Return a new DataFrame with the identified columns removed.
    return dataframe.drop(columns=cols_to_drop)


# Apply the function to the DataFrame to remove columns with too many missing values.
df = drop_high_missing_columns(df)


Dropping columns with more than 30% missing values: ['Alley', 'Fireplace Qu', 'Pool QC', 'Fence', 'Misc Feature']


## Drop Rows with Missing Target

In [4]:
def drop_missing_target(dataframe, target_column):
    """
    Drop rows from the DataFrame where the target variable is missing.

    This function checks if the target column exists in the DataFrame.
    If it does, rows with missing values in the target column are removed.
    This is crucial because missing target values can cause issues during model training,
    leading to inaccurate or biased predictions.

    Parameters:
        dataframe (pd.DataFrame): The input DataFrame to process.
        target_column (str): The name of the target variable column.

    Returns:
        pd.DataFrame: A DataFrame with rows removed where the target variable is missing.
    """
    # Check if the target column exists in the DataFrame.
    if target_column not in dataframe.columns:
        print(f"Target column '{target_column}' not found.")
        return dataframe

    # Record the number of rows before dropping missing target values.
    before = len(dataframe)
    
    # Drop rows where the target column has missing values.
    dataframe = dataframe.dropna(subset=[target_column])
    
    # Record the number of rows after the operation.
    after = len(dataframe)
    
    # Inform the user about how many rows were dropped.
    print(f"Dropped {before - after} rows with missing '{target_column}'.")
    
    return dataframe

# Apply the function to drop rows with missing values in the target column.
df = drop_missing_target(df, target_col)


Dropped 0 rows with missing 'SalePrice'.


## Fill Remaining Missing Values

In [5]:
def fill_missing_values(dataframe):
    """
    Fill missing values in the DataFrame for both numerical and categorical columns.

    For numerical columns, missing values are replaced with the median. The median is 
    chosen because it is less sensitive to outliers compared to the mean. For categorical 
    columns, missing values are replaced with the string "Missing" to explicitly denote 
    that data was absent.

    Parameters:
        dataframe (pd.DataFrame): The input DataFrame with missing values.

    Returns:
        pd.DataFrame: A new DataFrame with missing values filled in.
    """
    # Create a copy of the DataFrame to avoid modifying the original data.
    df_copy = dataframe.copy()

    # Identify numerical columns using pandas' select_dtypes method.
    numeric_cols = df_copy.select_dtypes(include=[np.number]).columns
    
    # Identify categorical columns (non-numeric) using select_dtypes.
    categorical_cols = df_copy.select_dtypes(exclude=[np.number]).columns

    # Fill missing values in numerical columns with the median value of each column.
    df_copy[numeric_cols] = df_copy[numeric_cols].fillna(df_copy[numeric_cols].median())
    
    # Fill missing values in categorical columns with the string 'Missing'.
    df_copy[categorical_cols] = df_copy[categorical_cols].fillna("Missing")
    
    return df_copy


# Apply the fill_missing_values function to the DataFrame.
df = fill_missing_values(df)


## Remove Outliers

In [6]:
def remove_outliers(dataframe, col_name, upper_limit):
    """
    Remove rows from the DataFrame where the values in the specified column exceed the upper_limit.

    Outlier removal is crucial to reduce the impact of extreme values that might skew model training.
    This function filters the DataFrame, keeping only those rows where the value in the given column
    is below the provided upper_limit. If the specified column is not found, it prints a message and returns
    the DataFrame unmodified.

    Parameters:
        dataframe (pd.DataFrame): The input DataFrame.
        col_name (str): The name of the column to inspect for outliers.
        upper_limit (float or int): The threshold value; rows with values equal to or above this will be removed.

    Returns:
        pd.DataFrame: The filtered DataFrame with outliers removed.
    """
    # Check if the specified column exists in the DataFrame.
    if col_name not in dataframe.columns:
        print(f"Column '{col_name}' not found. Skipping outlier removal.")
        return dataframe

    # Record the number of rows before removing outliers.
    before = len(dataframe)
    
    # Filter the DataFrame to keep rows where the column value is less than the upper limit.
    dataframe = dataframe[dataframe[col_name] < upper_limit]
    
    # Record the number of rows after filtering.
    after = len(dataframe)
    
    # Inform the user how many rows were removed.
    print(f"Removed {before - after} outliers from '{col_name}'.")
    
    return dataframe

# Apply the outlier removal function to the DataFrame.
df = remove_outliers(df, col_name="Gr Liv Area", upper_limit=4000)


Removed 5 outliers from 'Gr Liv Area'.


## Encode Categorical Variables

In [7]:
def encode_categorical_features(dataframe, freq_threshold=10):
    """
    Encode categorical features in the DataFrame using two strategies:
    
    - One-hot encoding for categorical variables with a number of unique categories
      less than or equal to freq_threshold. One-hot encoding creates binary columns for each
      category (dropping the first to avoid multicollinearity).
    
    - Frequency encoding for categorical variables with more than freq_threshold unique categories.
      Frequency encoding replaces each category with its relative frequency in the column.
    
    This approach helps manage high-cardinality features while still providing useful representations
    for variables with fewer categories.
    
    Parameters:
        dataframe (pd.DataFrame): The input DataFrame containing the data.
        freq_threshold (int): The maximum number of unique categories for which one-hot encoding is applied.
                              Categories with a count greater than this threshold will be frequency encoded.
    
    Returns:
        pd.DataFrame: A new DataFrame with categorical features encoded and the original categorical
                      columns removed.
    """
    # Create a copy of the DataFrame to avoid modifying the original.
    df_copy = dataframe.copy()
    
    # Identify categorical columns (columns with object type).
    cat_cols = df_copy.select_dtypes(include=["object"]).columns
    
    # List to store DataFrames from one-hot encoding.
    one_hot_frames = []
    
    # Dictionary to store frequency encoded columns.
    freq_frames = {}

    # Iterate over each categorical column to apply the appropriate encoding.
    for col in cat_cols:
        unique_count = df_copy[col].nunique()
        if unique_count > freq_threshold:
            # Frequency encoding: calculate normalized counts (relative frequency).
            freq_map = df_copy[col].value_counts(normalize=True)
            # Create a new column with the suffix '_freq' for frequency encoded values.
            freq_frames[col + "_freq"] = df_copy[col].map(freq_map)
        else:
            # One-hot encoding: create binary columns and drop the first category to avoid multicollinearity.
            one_hot_encoded = pd.get_dummies(df_copy[col], prefix=col, drop_first=True, dtype=int)
            one_hot_frames.append(one_hot_encoded)

    # Merge frequency encoded columns into the DataFrame if any exist.
    if freq_frames:
        freq_df = pd.DataFrame(freq_frames, index=df_copy.index)
        df_copy = df_copy.join(freq_df)
    
    # Merge one-hot encoded columns into the DataFrame if any exist.
    if one_hot_frames:
        one_hot_df = pd.concat(one_hot_frames, axis=1)
        df_copy = df_copy.join(one_hot_df)

    # Drop the original categorical columns since they are now encoded.
    df_copy = df_copy.drop(columns=cat_cols)
    
    return df_copy


# Apply the encoding function to the DataFrame.
df = encode_categorical_features(df)


# STEP 4: CREATE A BINARY TARGET

In [ ]:
# Calculate the median of the target variable.
# This median value acts as a threshold to classify observations.
median_value = df[target_name].median()

# Create a new binary column 'BinaryTarget' based on the median:
#   - 1 if the target value is above the median.
#   - 0 if the target value is at or below the median.
df["BinaryTarget"] = (df[target_name] > median_value).astype(int)

# Remove the original target column to simplify the dataset for classification.
df.drop(columns=[target_name], inplace=True)

# Update the target column reference for further processing.
target_col = "BinaryTarget"

# Define display labels for the binary classes
label_map = {0: f"Below Median {target_name}", 1: f"Above Median {target_name}"}

# Print the distribution of the new binary target classes
class_counts = df[target_col].value_counts().sort_index()
class_percentages = (class_counts / len(df)) * 100

print(f"\nMedian {target_name}: {median_value:,.2f}")
print(f"\nTarget Distribution:")
for category, count in class_counts.items():
    print(f"  {label_map[category]}: {count} samples ({class_percentages[category]:.2f}%)")

# Print a sample of the modified DataFrame
print("\nSample of the Modified Dataset:")
print(df.head())

# STEP 5: SPLIT THE DATA INTO TRAIN AND VALIDATION

In [9]:
# Separate the features (X) and the target variable (y).
# X contains all columns except the target column.
# y contains only the target column.
X = df.drop(columns=[target_col])  # Features (predictor variables)
y = df[target_col]  # Target variable

# Split the data into training and validation sets.
# 'test_size=0.2' means 20% of the data is reserved for validation, and 80% for training.
# 'random_state=42' ensures that the split remains the same each time the code is run.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes of the training and validation sets to confirm successful splitting.
print(f"Training set: Features {X_train.shape}, Target {y_train.shape}")
print(f"Validation set: Features {X_val.shape}, Target {y_val.shape}")


Training set: Features (2340, 203), Target (2340,)
Validation set: Features (585, 203), Target (585,)


# STEP 6: PREPARE FILES AND SET UP SAGEMAKER SESSION

In [10]:
# Initialize a SageMaker session
sagemaker_session = sagemaker.Session()

# Get IAM role associated with SageMaker
role = get_execution_role()

# Define S3 bucket, AWS region, and S3 folder prefix for storing data
bucket = sagemaker_session.default_bucket()  # Default S3 bucket assigned by SageMaker
region = sagemaker_session.boto_region_name  # AWS region of the session
prefix = "sagemaker/binary-ames-housing"  # Folder path in S3

# Combine target labels (y) and feature data (X) for training and validation sets
train_data = pd.concat([y_train, X_train], axis=1)
validation_data = pd.concat([y_val, X_val], axis=1)

# Check for missing values and print a warning if found
if train_data.isnull().values.any() or validation_data.isnull().values.any():
    print("Warning: Missing values detected! Consider handling them before training.")

# Check for class imbalance and print a warning if any class dominates (>75% of the data)
train_class_distribution = y_train.value_counts(normalize=True)
val_class_distribution = y_val.value_counts(normalize=True)

if train_class_distribution.max() > 0.75 or val_class_distribution.max() > 0.75:
    print("Warning: Class imbalance detected. Consider balancing the dataset.")

# Define filenames for local storage
train_file = "ames_binary_train.csv"
validation_file = "ames_binary_validation.csv"

# Save training and validation data to CSV files (no headers, no index)
train_data.to_csv(train_file, index=False, header=False)
validation_data.to_csv(validation_file, index=False, header=False)

# Upload datasets to Amazon S3 and retrieve their S3 locations
train_uri = sagemaker_session.upload_data(path=train_file, bucket=bucket, key_prefix=prefix)
validation_uri = sagemaker_session.upload_data(path=validation_file, bucket=bucket, key_prefix=prefix)

# Print the S3 paths of uploaded datasets for reference
print(f"Training data uploaded to: {train_uri}")
print(f"Validation data uploaded to: {validation_uri}")


[02/04/25 12:05:40] INFO     Found credentials from IAM Role:                                   ]8;id=218857;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=731031;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

[02/04/25 12:05:41] INFO     Found credentials from IAM Role:                                   ]8;id=304222;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=582169;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

Training data uploaded to: s3://sagemaker-us-east-1-785881939712/sagemaker/binary-ames-housing/ames_binary_train.csv
Validation data uploaded to: s3://sagemaker-us-east-1-785881939712/sagemaker/binary-ames-housing/ames_binary_validation.csv


# STEP 7: TRAIN THE LINEAR LEARNER MODEL WITH HYPERPARAMETER TUNING (BINARY CLASSIFICATION)

In [12]:
# Create TrainingInput objects for tuning
train_input = TrainingInput(s3_data=train_uri, content_type="text/csv")
validation_input = TrainingInput(s3_data=validation_uri, content_type="text/csv")

# Retrieve the container image URI for the Linear Learner algorithm in the specified region.
container = sagemaker.image_uris.retrieve(
    framework="linear-learner",
    region=region
)

# Determine the number of features (ensure this is defined using your dataset).
num_features = X.shape[1]

# Instantiate the SageMaker Estimator for the Linear Learner model.
linear_learner = Estimator(
    image_uri=container,                        # The container image for Linear Learner.
    role=role,                                  # IAM role with necessary permissions.
    instance_count=1,                           # Number of instances.
    instance_type='ml.m5.large',                # Instance type based on resource needs.
    output_path=f's3://{bucket}/{prefix}/output',# S3 bucket & prefix for model artifacts.
    sagemaker_session=sagemaker_session         # Active SageMaker session.
)

# Set base hyperparameters for the Linear Learner.
linear_learner.set_hyperparameters(
    feature_dim=num_features,
    predictor_type='binary_classifier'
)

# -------------------------------
# Hyperparameter Tuning Section
# -------------------------------

# Define hyperparameter ranges to tune.
hyperparameter_ranges = {
    "learning_rate": ContinuousParameter(0.001, 0.1),
}

# Specify the objective metric.
objective_metric_name = "validation:binary_classification_accuracy"
objective_type = "Maximize"

# Create the Hyperparameter Tuner.
tuner = HyperparameterTuner(
    estimator=linear_learner,
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    objective_type=objective_type,
    max_jobs=10,            # Total number of training jobs to be run.
    max_parallel_jobs=2,     # How many training jobs run in parallel.
)

# Create TrainingInput objects for tuning.
train_input = TrainingInput(s3_data=train_uri, content_type="text/csv")
validation_input = TrainingInput(s3_data=validation_uri, content_type="text/csv")

# Launch the hyperparameter tuning job.
tuner.fit({'train': train_input, 'validation': validation_input})
print("Hyperparameter tuning job launched!")


[02/04/25 12:08:45] INFO     Same images used for training and inference. Defaulting to image     ]8;id=700175;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=805427;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#391\391]8;;\
                             scope: inference.                                                                     

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=836827;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=31108;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#528\528]8;;\

                    WARNING  No finished training job found associated with this estimator.       ]8;id=780387;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py\estimator.py]8;;\:]8;id=80332;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py#1914\1914]8;;\
                             Please make sure this estimator is only used for building workflow                    
                             config                                                                                

                    WARNING  No finished training job found associated with this estimator.       ]8;id=930445;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py\estimator.py]8;;\:]8;id=111326;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py#1914\1914]8;;\
                             Please make sure this estimator is only used for building workflow                    
                             config                                                                                

                    INFO     Creating hyperparameter tuning job with name:                          ]8;id=603444;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=484417;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#3383\3383]8;;\
                             linear-learner-250204-1208                                                            

......................................................................................................!
Hyperparameter tuning job launched!


# STEP 8: DEPLOY THE BEST MODEL FROM THE TUNING JOB

In [ ]:
# -----------------------------------------------------------------------------
# Initialize session and SageMaker client.
# -----------------------------------------------------------------------------
session = Session()  # High-level SageMaker session
sm_client = boto3.client("sagemaker")

# endpoint_name is defined in the CONFIGURATION block in Step 1.

# -----------------------------------------------------------------------------
# 1. Built-in waiter for InService (helps waiting on creation)
# -----------------------------------------------------------------------------
endpoint_in_service_waiter = sm_client.get_waiter("endpoint_in_service")

def delete_endpoint_and_config(endpoint_name: str, wait_for_deletion: bool = True) -> None:
    """
    Deletes an endpoint and its corresponding endpoint configuration (if they exist).
    Optionally polls until resources are deleted.
    """
    # 1. Delete endpoint (if it exists).
    try:
        endpoint_desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
        endpoint_status = endpoint_desc["EndpointStatus"]

        # If the endpoint is Creating or Updating, wait until it becomes InService.
        if endpoint_status in ("Creating", "Updating"):
            logger.info(f"Endpoint '{endpoint_name}' is in '{endpoint_status}' state. Waiting before deletion.")
            endpoint_in_service_waiter.wait(EndpointName=endpoint_name)
        
        logger.info(f"Deleting endpoint: {endpoint_name}")
        sm_client.delete_endpoint(EndpointName=endpoint_name)

    except ClientError as e:
        if e.response["Error"]["Code"] == "ValidationException" and "Could not find" in e.response["Error"]["Message"]:
            logger.info(f"Endpoint '{endpoint_name}' does not exist or has already been deleted.")
        else:
            raise e

    # 2. Delete endpoint configuration (if it exists).
    try:
        sm_client.describe_endpoint_config(EndpointConfigName=endpoint_name)
        logger.info(f"Deleting endpoint configuration: {endpoint_name}")
        sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    except ClientError as e:
        if e.response["Error"]["Code"] == "ValidationException" and "Could not find" in e.response["Error"]["Message"]:
            logger.info(f"Endpoint config '{endpoint_name}' does not exist or has already been deleted.")
        else:
            raise e

    # 3. Optionally poll for deletion.
    if wait_for_deletion:
        logger.info("Waiting for endpoint & configuration to be deleted...")
        for _ in range(30):
            endpoint_exists = True
            endpoint_config_exists = True

            try:
                sm_client.describe_endpoint(EndpointName=endpoint_name)
            except ClientError as e:
                if "Could not find" in e.response["Error"]["Message"]:
                    endpoint_exists = False

            try:
                sm_client.describe_endpoint_config(EndpointConfigName=endpoint_name)
            except ClientError as e:
                if "Could not find" in e.response["Error"]["Message"]:
                    endpoint_config_exists = False

            if not endpoint_exists and not endpoint_config_exists:
                logger.info("Endpoint and endpoint config fully deleted.")
                break

            logger.info("Endpoint or endpoint config still deleting... sleeping 10s.")
            time.sleep(10)
        else:
            logger.warning("Endpoint or endpoint config not fully deleted after 30 checks.")

def delete_model(model_name: str, wait_for_deletion: bool = True) -> None:
    """
    Deletes a SageMaker model if it exists. Optionally waits until it disappears.
    """
    try:
        sm_client.describe_model(ModelName=model_name)
        logger.info(f"Deleting model: {model_name}")
        sm_client.delete_model(ModelName=model_name)
    except ClientError as e:
        if "Could not find" in e.response["Error"]["Message"]:
            logger.info(f"Model '{model_name}' does not exist or is already deleted.")
        else:
            raise e

    if wait_for_deletion:
        for _ in range(20):
            try:
                sm_client.describe_model(ModelName=model_name)
                logger.info("Model still deleting... sleeping 5s.")
                time.sleep(5)
            except ClientError as e:
                if "Could not find" in e.response["Error"]["Message"]:
                    logger.info("Model fully deleted.")
                    break
        else:
            logger.warning("Model was not deleted after waiting.")

# -----------------------------------------------------------------------------
# Delete any existing endpoint and configuration with the same name.
# -----------------------------------------------------------------------------
delete_endpoint_and_config(endpoint_name)

# -----------------------------------------------------------------------------
# Retrieve the best estimator from the completed tuning job.
# -----------------------------------------------------------------------------
best_estimator = tuner.best_estimator()  # Assumes that 'tuner' is defined and tuning is complete

# -----------------------------------------------------------------------------
# Deploy the best model.
# -----------------------------------------------------------------------------
predictor = best_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name
)

# Set the serializer and deserializer for the predictor.
predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

logger.info(f"Best model deployed to endpoint: {endpoint_name} and ready for inference.")

# STEP 9: EVALUATE THE DEPLOYED MODEL

In [ ]:
def evaluate_deployed_classifier(predictor, X_val, y_val):
    """
    Evaluate the deployed classification model on the test set.

    Parameters:
      predictor: SageMaker Predictor object for the deployed model.
      X_val (pd.DataFrame): Test features.
      y_val (pd.Series): True labels.

    Returns:
      dict: Evaluation metrics (Accuracy, Precision, Recall, F1 Score)
    """

    # Ensure the predictor uses the correct serialization
    predictor.serializer = CSVSerializer()

    # Copy test features and labels
    X_test = X_val.copy()
    y_test = y_val.copy()

    # Get predictions from the deployed endpoint
    predictions = predictor.predict(X_test.values.astype(np.float64))

    # Ensure correct extraction of predicted labels
    try:
        predicted_labels = [int(result["predicted_label"]) for result in predictions["predictions"]]
    except (KeyError, TypeError) as e:
        print(f"Error extracting predictions: {e}")
        return None  # Stop execution if predictions are not correctly formatted

    # Compute evaluation metrics
    acc = accuracy_score(y_test, predicted_labels)
    prec = precision_score(y_test, predicted_labels, zero_division=0)
    rec = recall_score(y_test, predicted_labels, zero_division=0)
    f1 = f1_score(y_test, predicted_labels, zero_division=0)

    # Print evaluation metrics
    print("\n **Evaluation Metrics:**")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}\n")

    # Print detailed classification report
    print("**Classification Report:**")
    print(classification_report(y_test, predicted_labels))

    # Compute and visualize the confusion matrix
    cm = confusion_matrix(y_test, predicted_labels)
    print("\n**Confusion Matrix (Raw Values):**")
    print(cm)

    # Confusion Matrix with percentages for better readability
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # Normalize per class
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_percent, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=[label_map[0], label_map[1]],
                yticklabels=[label_map[0], label_map[1]])
    plt.title("Confusion Matrix (Percentage)")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.show()

    # Return computed evaluation metrics
    return {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    }

# Example call:
# Replace 'predictor', 'X_val', and 'y_val' with actual objects
metrics = evaluate_deployed_classifier(predictor, X_val, y_val)
print(metrics)

# STEP 10: QUERY THE DEPLOYED ENDPOINT WITH TEST DATA

In [ ]:
def query_test_predictions(predictor, X_val, y_val, n_samples=5):
    """
    Query the deployed endpoint with test data and display results as a
    formatted DataFrame table with probability bar chart and accuracy summary.

    Parameters:
        predictor: The SageMaker Predictor object.
        X_val (pd.DataFrame): Validation features.
        y_val (pd.Series): True target values.
        n_samples (int): Number of samples to query.
    """
    # Select the first n rows from the validation set
    sample_data = X_val.head(n_samples)
    input_data = sample_data.values.astype(np.float64)

    # Request predictions from the deployed endpoint
    predictions = predictor.predict(input_data)

    if "predictions" not in predictions:
        print("No 'predictions' key found in the response:", predictions)
        return

    # Extract probabilities and compute predicted classes
    probabilities = [float(result["score"]) for result in predictions["predictions"]]
    threshold = 0.5
    predicted_classes = [1 if p > threshold else 0 for p in probabilities]
    actual_classes = y_val.loc[sample_data.index].tolist()
    results = ["CORRECT" if p == a else "INCORRECT" for p, a in zip(predicted_classes, actual_classes)]

    # Build a results DataFrame
    results_df = pd.DataFrame({
        "Sample": [f"Row {i}" for i in range(1, n_samples + 1)],
        "Probability": [f"{p:.4f}" for p in probabilities],
        "Predicted": [f"{c} ({label_map[c]})" for c in predicted_classes],
        "Actual": [f"{a} ({label_map[a]})" for a in actual_classes],
        "Result": results
    })

    print("\nTest Predictions on 5 Validation Samples:")
    display(results_df)

    # Accuracy summary
    n_correct = results.count("CORRECT")
    print(f"\nAccuracy: {n_correct} of {n_samples} correct ({n_correct/n_samples*100:.0f}%)")

    # Horizontal bar chart showing prediction probabilities with threshold line
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#55A868" if r == "CORRECT" else "#C44E52" for r in results]
    y_pos = np.arange(n_samples)
    bars = ax.barh(y_pos, probabilities, color=colors, height=0.6)

    # Add threshold line at 0.5
    ax.axvline(x=threshold, color="black", linestyle="--", linewidth=1.5, label=f"Threshold ({threshold})")

    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"Row {i}" for i in range(1, n_samples + 1)])
    ax.set_xlabel("Prediction Probability (Above Median)")
    ax.set_title("Prediction Probabilities with Classification Threshold")
    ax.set_xlim(0, 1)
    ax.legend()

    # Add predicted label text on each bar
    for i, (prob, pred_class) in enumerate(zip(probabilities, predicted_classes)):
        ax.text(prob + 0.02, i, f"{label_map[pred_class]}", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()

query_test_predictions(predictor, X_val, y_val)

# STEP 11: INTERACTIVE PREDICTION

Now it is your turn! Select any row from the validation set by entering its row number.
The model will predict whether the target value is above or below the median
and compare it to the actual classification.

Enter `quit` when you are done exploring.

In [ ]:
def interactive_prediction():
    """
    Allow students to select validation rows, predict binary class,
    and visualize the prediction probability relative to the threshold.
    """
    threshold = 0.5

    print(f"Validation set has {len(X_val)} rows (indices 0 to {len(X_val)-1})")
    print("Enter a row number to predict its class, or 'quit' to stop.\n")

    while True:
        user_input = input("Enter row number (or 'quit'): ").strip()
        if user_input.lower() == "quit":
            print("\nDone exploring predictions.")
            break
        try:
            idx = int(user_input)
            if idx < 0 or idx >= len(X_val):
                print(f"  Row number must be between 0 and {len(X_val)-1}\n")
                continue

            # Send the selected row to the endpoint
            row = X_val.iloc[idx:idx+1].values.astype(np.float64)
            result = predictor.predict(row)

            if "predictions" not in result:
                print("  No predictions key found in response.\n")
                continue

            prob = float(result["predictions"][0]["score"])
            predicted_class = 1 if prob > threshold else 0
            actual = int(y_val.iloc[idx])
            match = "CORRECT" if predicted_class == actual else "INCORRECT"

            # Print numeric results
            print(f"  Probability:      {prob:.4f}")
            print(f"  Predicted Class:  {predicted_class} ({label_map[predicted_class]})")
            print(f"  Actual Class:     {actual} ({label_map[actual]})")
            print(f"  Result:           {match}")

            # Probability bar chart with threshold marker
            fig, ax = plt.subplots(figsize=(8, 1.5))
            color = "#55A868" if match == "CORRECT" else "#C44E52"
            ax.barh([0], [prob], color=color, height=0.4)
            ax.axvline(x=threshold, color="black", linestyle="--", linewidth=1.5, label="Threshold (0.5)")
            ax.set_xlim(0, 1)
            ax.set_yticks([])
            ax.set_xlabel("Probability (Above Median)")
            ax.set_title(f"Row {idx}: {label_map[predicted_class]} (prob={prob:.4f}) - {match}")
            ax.legend(loc="upper right", fontsize=8)
            plt.tight_layout()
            plt.show()
            print()

        except ValueError:
            print("  Please enter a valid number or 'quit'.\n")

interactive_prediction()

# STEP 12: DELETE THE ENDPOINT AND ENDPOINT CONFIG

This step reuses the `delete_endpoint_and_config` function defined in Step 8 to clean up AWS resources.

You will be prompted to confirm before deletion.

In [ ]:
# ⚠️ AWS COST WARNING ⚠️
# SageMaker endpoints incur charges as long as they are running.
# Make sure to delete your endpoint when you are done to avoid unexpected costs.

response = input("Are you sure you want to delete the endpoint? (yes/no): ").strip().lower()
if response == "yes":
    delete_endpoint_and_config(endpoint_name)
    print("\nEndpoint cleanup complete.")
else:
    print("\nEndpoint was NOT deleted. Remember to delete it later to avoid charges.")